# BitWeaver: BERT-base Fake Quantization + Dynamic Programming Allocation

Notebook ini menjalankan eksperimen untuk makalah **Efficient Transformer Compression via Dynamic Programming-Based Mixed-Precision Bit-Width Allocation**.

Pipeline:
1. Load dataset GLUE SST-2 subset.
2. Fine-tune/evaluate BERT-base untuk text classification.
3. Ukur layer-wise sensitivity dengan fake quantization untuk bit-width `{2, 4, 8}`.
4. Jalankan Dynamic Programming, greedy, dan uniform baseline.
5. Export `sensitivity_table.csv` dan `comparison_table.csv` untuk bagian Results and Discussion.

Disarankan dijalankan di Kaggle dengan accelerator GPU.

In [19]:
# Kaggle biasanya sudah punya torch/sklearn/numpy/pandas.
# Jalankan cell ini kalau transformers/datasets/evaluate belum tersedia.
!pip -q install transformers datasets evaluate accelerate

In [20]:
import copy
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

OUTPUT_DIR = Path('/kaggle/working/bitweaver-bert-base-results') if Path('/kaggle/working').exists() else Path('../results/bert-base')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('output:', OUTPUT_DIR.resolve())

device: cuda
output: /kaggle/working/bitweaver-bert-base-results


## Configuration

`MAX_TRAIN_SAMPLES` dan `MAX_EVAL_SAMPLES` sengaja dibuat kecil agar eksperimen realistis untuk Kaggle notebook. Untuk hasil final yang lebih kuat, naikkan nilainya.

In [21]:
MODEL_NAME = 'bert-base-uncased'
BIT_OPTIONS = [2, 4, 8]
MAX_TRAIN_SAMPLES = 2000
MAX_EVAL_SAMPLES = 500
BATCH_SIZE = 16
FINE_TUNE_EPOCHS = 1
LEARNING_RATE = 2e-5

# Budget dalam satuan relative bit-cost. BERT-base punya 12 layer.
# Jika tiap layer dianggap setara, all-8-bit = 96, all-4-bit = 48.
BUDGETS = [48, 56, 60, 64, 72]

## Dataset and Model

In [22]:
raw = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True)

encoded = raw.map(tokenize, batched=True)
encoded = encoded.rename_column('label', 'labels')
encoded.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_ds = encoded['train'].shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))
eval_ds = encoded['validation'].shuffle(seed=SEED).select(range(MAX_EVAL_SAMPLES))

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
print(model.config)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [23]:
def evaluate_model(model, loader):
    model.eval()
    preds, labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            total_loss += out.loss.item() * batch['labels'].size(0)
            pred = out.logits.argmax(dim=-1)
            preds.extend(pred.cpu().numpy().tolist())
            labels.extend(batch['labels'].cpu().numpy().tolist())
    return {
        'loss': total_loss / len(labels),
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds),
    }

def fine_tune(model, train_loader, epochs=1):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(epochs):
        started = time.time()
        total = 0.0
        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
            total += out.loss.item()
            if step % 50 == 0:
                print(f'epoch={epoch+1} step={step} avg_loss={total/step:.4f}')
        print(f'epoch {epoch+1} done in {time.time()-started:.1f}s, avg_loss={total/len(train_loader):.4f}')
    return model

baseline_before = evaluate_model(model, eval_loader)
print('baseline before fine-tune:', baseline_before)
model = fine_tune(model, train_loader, FINE_TUNE_EPOCHS)
baseline = evaluate_model(model, eval_loader)
print('baseline after fine-tune:', baseline)

baseline before fine-tune: {'loss': 0.6934241838455201, 'accuracy': 0.506, 'f1': 0.6425470332850941}
epoch=1 step=50 avg_loss=0.5774
epoch=1 step=100 avg_loss=0.4818
epoch 1 done in 16.4s, avg_loss=0.4532
baseline after fine-tune: {'loss': 0.27021011662483213, 'accuracy': 0.894, 'f1': 0.8978805394990366}


## Fake Quantization

Fake quantization membulatkan bobot ke level bit rendah lalu mengembalikannya ke float. Ini mengukur dampak precision loss tanpa membutuhkan custom low-bit inference kernel.

In [24]:
def fake_quantize_tensor(tensor, bits):
    if bits >= 32:
        return tensor.clone()
    if bits < 2:
        raise ValueError('bits must be >= 2')
    max_abs = tensor.detach().abs().max()
    if max_abs == 0:
        return tensor.clone()
    qmax = (2 ** (bits - 1)) - 1
    scale = max_abs / qmax
    quantized = torch.clamp(torch.round(tensor / scale), -qmax, qmax)
    return quantized * scale

def quantize_layer_inplace(layer, bits):
    original = {name: param.detach().clone() for name, param in layer.named_parameters() if param.requires_grad}
    with torch.no_grad():
        for name, param in layer.named_parameters():
            if param.requires_grad:
                param.copy_(fake_quantize_tensor(param, bits))
    return original

def restore_layer_inplace(layer, original):
    with torch.no_grad():
        for name, param in layer.named_parameters():
            if name in original:
                param.copy_(original[name])

def layer_parameter_count(layer):
    return sum(p.numel() for p in layer.parameters() if p.requires_grad)

layers = list(model.bert.encoder.layer)
print('num transformer layers:', len(layers))
print('params per layer:', [layer_parameter_count(layer) for layer in layers])

num transformer layers: 12
params per layer: [7087872, 7087872, 7087872, 7087872, 7087872, 7087872, 7087872, 7087872, 7087872, 7087872, 7087872, 7087872]


## Layer-wise Sensitivity Table

In [25]:
rows = []
base_acc = baseline['accuracy']
base_f1 = baseline['f1']

for layer_idx, layer in enumerate(layers):
    params = layer_parameter_count(layer)
    for bits in BIT_OPTIONS:
        original = quantize_layer_inplace(layer, bits)
        metrics = evaluate_model(model, eval_loader)
        restore_layer_inplace(layer, original)
        rows.append({
            'layer': layer_idx,
            'bits': bits,
            'params': params,
            'relative_size': bits,
            'bit_storage': params * bits,
            'accuracy': metrics['accuracy'],
            'f1': metrics['f1'],
            'accuracy_loss': max(0.0, base_acc - metrics['accuracy']),
            'f1_loss': max(0.0, base_f1 - metrics['f1']),
        })
        print(rows[-1])

sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(OUTPUT_DIR / 'sensitivity_table.csv', index=False)
sensitivity

{'layer': 0, 'bits': 2, 'params': 7087872, 'relative_size': 2, 'bit_storage': 14175744, 'accuracy': 0.478, 'f1': 0.0, 'accuracy_loss': 0.41600000000000004, 'f1_loss': 0.8978805394990366}
{'layer': 0, 'bits': 4, 'params': 7087872, 'relative_size': 4, 'bit_storage': 28351488, 'accuracy': 0.888, 'f1': 0.89272030651341, 'accuracy_loss': 0.006000000000000005, 'f1_loss': 0.005160232985626623}
{'layer': 0, 'bits': 8, 'params': 7087872, 'relative_size': 8, 'bit_storage': 56702976, 'accuracy': 0.894, 'f1': 0.8978805394990366, 'accuracy_loss': 0.0, 'f1_loss': 0.0}
{'layer': 1, 'bits': 2, 'params': 7087872, 'relative_size': 2, 'bit_storage': 14175744, 'accuracy': 0.472, 'f1': 0.12582781456953643, 'accuracy_loss': 0.42200000000000004, 'f1_loss': 0.7720527249295002}
{'layer': 1, 'bits': 4, 'params': 7087872, 'relative_size': 4, 'bit_storage': 28351488, 'accuracy': 0.898, 'f1': 0.9009708737864077, 'accuracy_loss': 0.0, 'f1_loss': 0.0}
{'layer': 1, 'bits': 8, 'params': 7087872, 'relative_size': 8, 'b

,layer,bits,params,relative_size,bit_storage,accuracy,f1,accuracy_loss,f1_loss
0,0,2,7087872,2,14175744,0.478,0.000000,0.416,0.897881
1,0,4,7087872,4,28351488,0.888,0.892720,0.006,0.005160
2,0,8,7087872,8,56702976,0.894,0.897881,0.000,0.000000
3,1,2,7087872,2,14175744,0.472,0.125828,0.422,0.772053
4,1,4,7087872,4,28351488,0.898,0.900971,0.000,0.000000
5,1,8,7087872,8,56702976,0.894,0.897881,0.000,0.000000
6,2,2,7087872,2,14175744,0.478,0.000000,0.416,0.897881
7,2,4,7087872,4,28351488,0.872,0.875486,0.022,0.022394
8,2,8,7087872,8,56702976,0.892,0.896552,0.002,0.001329
9,3,2,7087872,2,14175744,0.524,0.686842,0.370,0.211038


## Allocators

In [26]:
@dataclass
class AllocationResult:
    method: str
    budget: int
    config: list
    total_size: int
    total_loss: float
    feasible: bool = True

def table_to_options(df, loss_col='accuracy_loss', size_col='relative_size'):
    options = []
    for layer in sorted(df['layer'].unique()):
        layer_df = df[df['layer'] == layer]
        options.append({
            int(row.bits): {'size': int(getattr(row, size_col)), 'loss': float(getattr(row, loss_col))}
            for row in layer_df.itertuples(index=False)
        })
    return options

def score_config(options, config, method, budget):
    size = int(sum(options[i][bit]['size'] for i, bit in enumerate(config)))
    loss = float(sum(options[i][bit]['loss'] for i, bit in enumerate(config)))
    return AllocationResult(method, budget, list(config), size, loss, size <= budget)

def dp_allocate(options, budget):
    n = len(options)
    dp = [[math.inf] * (budget + 1) for _ in range(n + 1)]
    parent = [[None] * (budget + 1) for _ in range(n + 1)]
    prev = [[None] * (budget + 1) for _ in range(n + 1)]
    for b in range(budget + 1):
        dp[0][b] = 0.0
    for i, layer in enumerate(options, start=1):
        for b in range(budget + 1):
            for bit, values in layer.items():
                size = values['size']
                if size <= b:
                    cand = dp[i-1][b-size] + values['loss']
                    if cand < dp[i][b]:
                        dp[i][b] = cand
                        parent[i][b] = bit
                        prev[i][b] = b - size
    best_b = min(range(budget + 1), key=lambda b: dp[n][b])
    if dp[n][best_b] == math.inf:
        return AllocationResult('DP', budget, [], 0, math.inf, False)
    config = []
    b = best_b
    for i in range(n, 0, -1):
        bit = parent[i][b]
        config.append(bit)
        b = prev[i][b]
    config.reverse()
    return score_config(options, config, 'DP', budget)

def uniform_allocate(options, bit, budget):
    return score_config(options, [bit] * len(options), f'Uniform {bit}-bit', budget)

def greedy_allocate(options, budget):
    config = [max(layer.keys()) for layer in options]
    while score_config(options, config, 'Greedy', budget).total_size > budget:
        best = None
        for i, layer in enumerate(options):
            cur = config[i]
            lower = [b for b in layer if b < cur]
            if not lower:
                continue
            nxt = max(lower)
            saved = layer[cur]['size'] - layer[nxt]['size']
            added_loss = layer[nxt]['loss'] - layer[cur]['loss']
            candidate = (added_loss / saved, i, nxt)
            if best is None or candidate < best:
                best = candidate
        if best is None:
            break
        _, i, nxt = best
        config[i] = nxt
    return score_config(options, config, 'Greedy', budget)

options = table_to_options(sensitivity)
options

[{2: {'size': 2, 'loss': 0.41600000000000004},
  4: {'size': 4, 'loss': 0.006000000000000005},
  8: {'size': 8, 'loss': 0.0}},
 {2: {'size': 2, 'loss': 0.42200000000000004},
  4: {'size': 4, 'loss': 0.0},
  8: {'size': 8, 'loss': 0.0}},
 {2: {'size': 2, 'loss': 0.41600000000000004},
  4: {'size': 4, 'loss': 0.02200000000000002},
  8: {'size': 8, 'loss': 0.0020000000000000018}},
 {2: {'size': 2, 'loss': 0.37},
  4: {'size': 4, 'loss': 0.01200000000000001},
  8: {'size': 8, 'loss': 0.0}},
 {2: {'size': 2, 'loss': 0.41600000000000004},
  4: {'size': 4, 'loss': 0.01200000000000001},
  8: {'size': 8, 'loss': 0.010000000000000009}},
 {2: {'size': 2, 'loss': 0.41600000000000004},
  4: {'size': 4, 'loss': 0.03400000000000003},
  8: {'size': 8, 'loss': 0.0}},
 {2: {'size': 2, 'loss': 0.368},
  4: {'size': 4, 'loss': 0.014000000000000012},
  8: {'size': 8, 'loss': 0.0}},
 {2: {'size': 2, 'loss': 0.41600000000000004},
  4: {'size': 4, 'loss': 0.028000000000000025},
  8: {'size': 8, 'loss': 0.0040

In [27]:
results = []
for budget in BUDGETS:
    candidates = [dp_allocate(options, budget), greedy_allocate(options, budget)]
    candidates.extend(uniform_allocate(options, bit, budget) for bit in BIT_OPTIONS)
    for r in candidates:
        full_32_size = 32 * len(options)
        compression_ratio = full_32_size / r.total_size if r.total_size else math.inf
        results.append({
            'method': r.method,
            'budget': r.budget,
            'config': str(r.config),
            'total_size': r.total_size,
            'compression_ratio_vs_32bit': compression_ratio,
            'estimated_accuracy_loss': r.total_loss,
            'feasible': r.feasible,
        })

comparison = pd.DataFrame(results)
comparison.to_csv(OUTPUT_DIR / 'comparison_table.csv', index=False)
comparison

,method,budget,config,total_size,compression_ratio_vs_32bit,estimated_accuracy_loss,feasible
0,DP,48,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",48,8.000000,0.162,True
1,Greedy,48,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2]",46,8.347826,0.170,True
2,Uniform 2-bit,48,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",24,16.000000,4.462,True
3,Uniform 4-bit,48,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",48,8.000000,0.162,True
4,Uniform 8-bit,48,"[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]",96,4.000000,0.016,False
5,DP,56,"[4, 4, 4, 4, 4, 8, 4, 8, 4, 4, 4, 4]",56,6.857143,0.104,True
6,Greedy,56,"[4, 4, 4, 4, 4, 8, 4, 8, 4, 4, 4, 2]",54,7.111111,0.112,True
7,Uniform 2-bit,56,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",24,16.000000,4.462,True
8,Uniform 4-bit,56,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",48,8.000000,0.162,True
9,Uniform 8-bit,56,"[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]",96,4.000000,0.016,False


## Paper-ready Summary

In [28]:
print('Baseline:', baseline)
print('\nSensitivity table saved to:', OUTPUT_DIR / 'sensitivity_table.csv')
print('Comparison table saved to:', OUTPUT_DIR / 'comparison_table.csv')
display(sensitivity.pivot(index='layer', columns='bits', values='accuracy_loss'))
display(comparison[comparison['feasible']].sort_values(['budget', 'estimated_accuracy_loss']))

Baseline: {'loss': 0.27021011662483213, 'accuracy': 0.894, 'f1': 0.8978805394990366}

Sensitivity table saved to: /kaggle/working/bitweaver-bert-base-results/sensitivity_table.csv
Comparison table saved to: /kaggle/working/bitweaver-bert-base-results/comparison_table.csv


bits,2,4,8
layer,,,
0,0.416,0.006,0.000
1,0.422,0.000,0.000
2,0.416,0.022,0.002
3,0.370,0.012,0.000
4,0.416,0.012,0.010
5,0.416,0.034,0.000
6,0.368,0.014,0.000
7,0.416,0.028,0.004
8,0.372,0.014,0.000


,method,budget,config,total_size,compression_ratio_vs_32bit,estimated_accuracy_loss,feasible
0,DP,48,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",48,8.000000,0.162,True
3,Uniform 4-bit,48,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",48,8.000000,0.162,True
1,Greedy,48,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2]",46,8.347826,0.170,True
2,Uniform 2-bit,48,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",24,16.000000,4.462,True
5,DP,56,"[4, 4, 4, 4, 4, 8, 4, 8, 4, 4, 4, 4]",56,6.857143,0.104,True
6,Greedy,56,"[4, 4, 4, 4, 4, 8, 4, 8, 4, 4, 4, 2]",54,7.111111,0.112,True
8,Uniform 4-bit,56,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]",48,8.000000,0.162,True
7,Uniform 2-bit,56,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",24,16.000000,4.462,True
10,DP,60,"[4, 4, 8, 4, 4, 8, 4, 8, 4, 4, 4, 4]",60,6.400000,0.084,True
11,Greedy,60,"[4, 4, 8, 4, 4, 8, 4, 8, 4, 4, 4, 4]",60,6.400000,0.084,True
